## 上下文 -

### editing notes

当修改涉及逐步构建的函数时，除了改最终函数定义，还要回溯改所有中间探索步骤——单行测试、循环原型、`Video(...)` 构造等——凡是涉及被改字段/参数的 cell 都要同步更新，保持探索轨迹和最终代码一致。


### docs

### styling

### directory

In [ ]:
!tree

.
├── CONTROLLER.ipynb
├── LICENSE
├── MANIFEST.in
├── README.md
├── _proc
│   ├── 00_core.ipynb
│   ├── _docs
│   │   ├── index.html
│   │   ├── robots.txt
│   │   └── sitemap.xml
│   ├── _quarto.yml
│   ├── index.ipynb
│   ├── nbdev.yml
│   └── styles.css
├── cachy.jsonl
├── db.db
├── db.db-shm
├── db.db-wal
├── nbs
│   ├── 00_core.ipynb
│   ├── _quarto.yml
│   ├── db.db
│   ├── db.db-shm
│   ├── db.db-wal
│   ├── index.ipynb
│   ├── nbdev.yml
│   ├── styles.css
│   └── test.jpg
├── pyproject.toml
├── vlm_monitor
│   ├── __init__.py
│   ├── __pycache__
│   │   ├── __init__.cpython-312.pyc
│   │   └── core.cpython-312.pyc
│   ├── _modidx.py
│   └── core.py
└── vlm_monitor.egg-info
    ├── PKG-INFO
    ├── SOURCES.txt
    ├── dependency_links.txt
    ├── entry_points.txt
    ├── requires.txt
    └── top_level.txt

7 directories, 37 files


In [ ]:
!tree -L 1 ../data/timss

../data/timss
├── M-AU1
├── M-AU2
├── M-AU3
├── M-AU4
├── M-CZ1
├── M-CZ2
├── M-CZ3
├── M-CZ4
├── M-HK1
├── M-HK2
├── M-HK3
├── M-HK4
├── M-JP1
├── M-JP2
├── M-JP3
├── M-JP4
├── M-NL1
├── M-NL2
├── M-NL3
├── M-NL4
├── M-SW1
├── M-SW2
├── M-SW3
├── M-SW4
├── M-US1
├── M-US2
├── M-US3
└── M-US4

29 directories, 0 files


# controller

> Control the library from here

Hi there, in all interactions in this dialogue, I want you to refreshingly and invigoratingly guide me Polya style. At the same time, I don't want you to jump ahead of where I am, which includes asking follow-up questions. Simply acknowledge this, that's all.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

收到啦！(｀・ω・´)ゞ

我会用 Polya 的方式来引导你，一步一步走，绝不抢跑～

```json {.usage}
{"model": "accounts/fireworks/models/glm-5p2", "prompt_tokens": 50241, "completion_tokens": 178, "total_tokens": 50419, "cached_tokens": 39983, "cost": 0.03111303}
```


In [ ]:
#| hide
from nbdev import nbdev_export; nbdev_export('./nbs/')

from toolslm.xml import nb2xml
core = nb2xml(Path('./nbs/00_core.ipynb')); core

'<notebook><raw id="7bdca9ea"><source>---\nskip_exec: true\n---</raw><md id="277a059d"><source># core\n\n> Fill in a module description here</md><code id="31c24a3c"><source>#| hide\n%load_ext autoreload\n%autoreload 2</code><code id="d5246d1d"><source>#| default_exp core</code><md id="d2648116"><source>## Database</md><code id="1654310c"><source>from fastcore.all import *</code><code id="22ed8677"><source>class Video: id:int; title:str=\'\'; overview:str=\'\'; transcript:str=\'\'; length:int=0; sample_rate:int=0; path:str=\'\'\nclass Frame: id:int; video_id:int; frame_number:int; subtitle:str=\'\'\nclass Run: id:int; deploy_time:str; finish_time:str; start_time:str; total_duration:str; video_id:int; model:str; usage:str; num_frames:int; description:str\nclass RunFrame: run_id:int; frame_id:int; type:str; system_prompt:str; prompt:str; description:str; usage:str</code><code id="83cc5be2"><source>from fastlite import *</code><code id="a2e9a927"><source>!rm db.db\ndb = database(\'db.db\')

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:


  %reload_ext autoreload


In [ ]:
from vlm_monitor.core import *

In [ ]:
# !rm db.db
db = init_db(); db

<Database <apsw.Connection "/app/data/vlm-monitor/db.db" at 0x72edb43587c0>>

In [ ]:
dpath = Path('../data/timss/'); dpath

Path('../data/timss')

In [ ]:
dpaths = filter_paths(dpath.ls()).sorted(lambda o: (o.stem[:-1], o.stem[-1])); dpaths[:5]

[Path('../data/timss/M-AU1'), Path('../data/timss/M-AU2'), Path('../data/timss/M-AU3'), Path('../data/timss/M-AU4'), Path('../data/timss/M-CZ1')]

In [ ]:
# populate_db(db, dpaths)

In [ ]:
len(db.t.video()), len(db.t.frame())

(28, 80308)

In [ ]:
caveman_prompt = '''
Respond terse like smart caveman. All technical substance stay. Only fluff die.

## Persistence

ACTIVE EVERY RESPONSE. No revert after many turns. No filler drift. Still active if unsure.

## Rules

Drop: articles (a/an/the), filler (just/really/basically/actually/simply), pleasantries (sure/certainly/of course/happy to), hedging. Fragments OK. Short synonyms (big not extensive, fix not "implement a solution for"). Technical terms exact. Code blocks unchanged. Errors quoted exact.

Pattern: `[thing] [action] [reason]. [next step].`

Not: "Sure! I'd be happy to help you with that. The issue you're experiencing is likely caused by..."
Yes: "Bug in auth middleware. Token expiry check use `<` not `<=`. Fix:"

## Intensity

Example — "Why React component re-render?"
- "New object ref each render. Inline object prop = new ref = re-render. Wrap in `useMemo`."

Example — "Explain database connection pooling."
- "Pool reuse open DB connections. No new connection per request. Skip handshake overhead."

## Auto-Clarity

Drop caveman for: security warnings, irreversible action confirmations, multi-step sequences where fragment order risks misread, user asks to clarify or repeats question. Resume caveman after clear part done.

Example — destructive op:
> **Warning:** This will permanently delete all rows in the `users` table and cannot be undone.
> ```sql
> DROP TABLE users;
> ```
> Caveman resume. Verify backup exist first.
'''

In [ ]:
prompt_template = '{} Report only what is directly observable — no speculation, no assumptions, no decorative language, and no observations beyond what was specifically requested.'

In [ ]:
prompts = AttrDict(
    environment=prompt_template.format('Detail the environment presented in the given frame.'),
    blackboard=prompt_template.format('Detail the contents present on chalkboard/blackboard/whiteboard presented in the given frame.'),
    teacher=prompt_template.format("Detail the teacher presented in the given frame, including but not limited to, the teacher's expressions, emotions, gestures, and interactions/dynamics if any."),
    students=prompt_template.format("Detail the students presented in the given frame, including but not limited to, the students' expressions, emotions, gestures, and interactions/dynamics between each other if any."),
)

In [ ]:
models = AttrDict(
    gemma26b       =AttrDict(name='google/gemma-4-26b-a4b-it:free', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    gemma31b       =AttrDict(name='google/gemma-4-31b-it', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    grok4p3        =AttrDict(name='x-ai/grok-4.3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    grok4p5        =AttrDict(name='x-ai/grok-4.5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    qwen3p7plus    =AttrDict(name='qwen/qwen3.7-plus', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    qwen3p7max     =AttrDict(name='qwen/qwen3.7-max', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mistral3p5     =AttrDict(name='mistralai/mistral-medium-3-5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mistral4       =AttrDict(name='mistralai/mistral-small-2603', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    n2mini         =AttrDict(name='nex-agi/nex-n2-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    n2pro          =AttrDict(name='nex-agi/nex-n2-pro', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    hy3            =AttrDict(name='tencent/hy3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    fugu           =AttrDict(name='sakana/fugu-ultra', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    glm4p6v        =AttrDict(name='z-ai/glm-4.6v', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    glm5p2         =AttrDict(name='z-ai/glm-5.2', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi2p6        =AttrDict(name='moonshotai/kimi-k2.6', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi2p7code    =AttrDict(name='moonshotai/kimi-k2.7-code', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_nano  =AttrDict(name='nvidia/nemotron-3-nano-30b-a3b', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_omni  =AttrDict(name='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_ultra =AttrDict(name='nvidia/nemotron-3-ultra-550b-a55b', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    minimax_m3     =AttrDict(name='minimax/minimax-m3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    step3p7        =AttrDict(name='stepfun/step-3.7-flash', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mimo_v2p5      =AttrDict(name='xiaomi/mimo-v2.5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mimo_v2p5pro   =AttrDict(name='xiaomi/mimo-v2.5-pro', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    seed2p0mini    =AttrDict(name='bytedance-seed/seed-2.0-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    seed2p0lite    =AttrDict(name='bytedance-seed/seed-2.0-lite', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
)

In [ ]:
s = session(system=caveman_prompt, model=models.seed2p0mini.name, display=False, **models.seed2p0mini.kw)

In [ ]:
from cachy import enable_cachy, disable_cachy; enable_cachy()
r = await s([user('hi')]); r

Hi. State question or need.

<details markdown='1'>

- model: `bytedance-seed/seed-2.0-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=388, completion_tokens=104, total_tokens=492, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=97, raw={'prompt_tokens': 388, 'completion_tokens': 104, 'total_tokens': 492, 'cost': 8.04e-05, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 8.04e-05, 'upstream_inference_prompt_cost': 3.88e-05, 'upstream_inference_completions_cost': 4.16e-05}, 'completion_tokens_details': {'reasoning_tokens': 97, 'image_tokens': 0, 'audio_tokens': 0}})`

</details>

In [ ]:
# from nbdev import nbdev_export; nbdev_export('./nbs/')
# from vlm_monitor.core import *

In [ ]:
r = await deploy_run(db.t.video[1].id, db, s, prompts.environment, prompt_type='environment', step=3, stop=30, cache=True)

!! Using cache


╭─ Run #6 ═══════════════════════════════╮
│ Model    bytedance-seed/seed-2.0-mini
│ Start    0
│ Stop     30
│ Step     3
│ Frames   10
│ Cache    True
│ Time     15:10:41
╰──────────────────────────────────────────────╯


HTML(
<style>
    progress { appearance: none; border: none; border-radius: 4px; width: 300px;
        height: 20px; vertical-align: middle; background: #e0e0e0; }

    progress::-webkit-progress-bar { background: #e0e0e0; border-radius: 4px; }
    progress::-webkit-progress-value { background: #2196F3; border-radius: 4px; }
    progress::-moz-progress-bar { background: #2196F3; border-radius: 4px; }

    progress:not([value]) {
        background: repeating-linear-gradient(45deg, #7e7e7e, #7e7e7e 10px, #5c5c5c 10px, #5c5c5c 20px); }

    progress.progress-bar-interrupted::-webkit-progress-value { background: #F44336; }
    progress.progress-bar-interrupted::-moz-progress-value { background: #F44336; }
    progress.progress-bar-interrupted::-webkit-progress-bar { background: #F44336; }
    progress.progress-bar-interrupted::-moz-progress-bar { background: #F44336; }
    progress.progress-bar-interrupted { background: #F44336; }    

    table.fastprogress { border-collapse: collapse; margin: 1em 0; font-size: 0.9em; }
    table.fastprogress th, table.fastprogress td { padding: 8px 12px; border: 1px solid #ddd; text-align: left; }
    table.fastprogress thead tr { background: #f8f9fa; font-weight: bold; }
    table.fastprogress tbody tr:nth-of-type(even) { background: #f8f9fa; }
</style>
)

<div></div>

╭─ Run #6 Complete ═════════════════════╮
│ Finish   15:11:51
│ Elapsed  0:01:09
│ Cost     $0.0044 (HKD 0.03)
╰──────────────────────────────────────────────╯


i could potentially use fastcore.parallel here to run these all in tandem

## Time and Cost Calculations

| Model          | Step | 15min $ | 15min ⏱ | 30min $ | 30min ⏱ | 45min $ | 45min ⏱  | 1hr $  | 1hr ⏱    |
| -------------- | ---- | ------- | ------- | ------- | ------- | ------- | -------- | ------ | -------- |
| seed2.0-mini   | s3   | $0.38   | 1h20m   | $0.76   | 2h41m   | $1.14   | 4h02m    | $1.52  | 5h22m    |
| seed2.0-mini   | s2   | $0.59   | 2h05m   | $1.17   | 4h09m   | $1.76   | 6h14m    | $2.34  | 8h18m    |
| kimi-k3        | s2   | $13.69  | 11h34m  | $27.37  | 23h08m  | $41.06  | 1d10h42m | $54.74 | 1d22h16m |
| kimi-k3        | s3   | $8.35   | 7h17m   | $16.70  | 14h33m  | $25.05  | 21h50m   | $33.40 | 1d05h06m |
| gemma-4-31b    | s2   | $0.41   | 9h00m   | $0.82   | 18h00m  | $1.23   | 1d03h00m | $1.64  | 1d12h00m |
| gemma-4-31b    | s3   | $0.24   | 5h26m   | $0.49   | 10h52m  | $0.73   | 16h18m   | $0.97  | 21h44m   |
| qwen3.7-plus   | s3   | $1.33   | 4h49m   | $2.65   | 9h37m   | $3.98   | 14h26m   | $5.30  | 19h14m   |
| glm-4.6v       | s2   | $0.94   | 4h51m   | $1.87   | 9h41m   | $2.81   | 14h32m   | $3.74  | 19h22m   |
| glm-4.6v       | s3   | $0.51   | 2h48m   | $1.03   | 5h36m   | $1.54   | 8h24m    | $2.05  | 11h12m   |
| minimax-m3     | s2   | $0.87   | 3h19m   | $1.74   | 6h38m   | $2.61   | 9h57m    | $3.48  | 13h16m   |
| minimax-m3     | s3   | $0.53   | 2h11m   | $1.05   | 4h21m   | $1.58   | 6h32m    | $2.10  | 8h42m    |
| step-3.7-flash | s2   | $2.06   | 3h54m   | $4.11   | 7h48m   | $6.17   | 11h42m   | $8.22  | 15h36m   |
| step-3.7-flash | s3   | $1.76   | 4h11m   | $3.52   | 8h22m   | $5.27   | 12h33m   | $7.03  | 16h44m   |
| mimo-v2.5      | s3   | $0.26   | 8h18m   | $0.51   | 16h35m  | $0.77   | 1d00h53m | $1.02  | 1d09h10m |
| mistral-small  | s2   | $0.35   | 1h25m   | $0.69   | 2h50m   | $1.04   | 4h15m    | $1.38  | 5h40m    |
| mistral-small  | s3   | $0.21   | 54m     | $0.43   | 1h48m   | $0.64   | 2h42m    | $0.85  | 3h36m    |
| grok-4.5       | s2   | $6.77   | 5h32m   | $13.54  | 11h04m  | $20.30  | 16h36m   | $27.07 | 22h08m   |
| grok-4.5       | s3   | $3.96   | 3h37m   | $7.91   | 7h13m   | $11.87  | 10h50m   | $15.83 | 14h26m   |

Excluded the two errored runs (qwen s2, mimo s2).

Cost winners: **mistral-small s3** ($0.21–$0.85) and **gemma-4-31b s3** ($0.24–$0.97) dominate. Speed winner: **mistral-small s3** (54min–3h36m). kimi-k3 is wildly expensive ($8–$55) and slow. grok-4.5 also pricey ($4–$27). Note step-3.7-flash s3 is *slower* than s2 despite fewer frames — interesting per-frame overhead.

1. **从每次成功的运行输出中提取数据** — 我查看了每个运行的 `Frames`、`Cost` 和 `Elapsed` 数值，忽略了两次失败的情况（qwen s2 和 mimo s2）。
2. **计算每帧的成本 (cpf)** — `cpf = cost / frames`。例如，mistral-small s3：$0.0071 / 10 = $0.00071 per frame。
3. **计算每帧的时间 (tpf)** — `tpf = elapsed_seconds / frames`。例如，mistral-small s3：108s / 10 = 10.8s per frame。
4. **根据步长确定视频的帧数** — 在 `sample_rate=1`（每秒 1 帧）的情况下，15 分钟的视频有 900 帧。在 `step=3` 时，处理 `900 // 3 = 300` 帧。在 `step=2` 时，处理 `900 // 2 = 450` 帧。
5. **乘以估计值** — `cost = num_frames × cpf`，`time = num_frames × tpf`，分别针对 15 分钟（900s）、30 分钟（1800s）、45 分钟（2700s）、1 小时（3600s）的视频进行计算。

一个假设是：各模型之间的 **每帧成本和时间保持不变** — 即扩展是线性的。这是一个一阶近似值；实际运行可能会有所不同，原因包括 API 延迟波动、缓存效应或内容触发的速率限制（这正是导致 qwen 和 mimo 失败的原因）。